In [1]:
import mo_gymnasium as mo_gym
import numpy as np
import envs
import csv
from src.modules.commun import Constant
from src.modules.Tools import Tools
#from src.scripts.delete_wandb_dir import delete_directory
from morl_baselines.multi_policy.multi_policy_moqlearning.mp_mo_q_learning import (
    MPMOQLearning,
)

from morl_baselines.multi_policy.pcn.pcn import PCN

GAMMA = 1


# IMPORTANT____________________________________________________________________________________________
# Init Env by giving it the service querry to optimize + the folder where to find the PREPROCESSED data 
serviceIds = [0 , 1 , 2 ]
number_services = len(serviceIds)
MultiCloud_data_dir="./src/data/preprocessedData/NC_20_NS_50/NC_20_NS_50_01"
#______________________________________________________________________________________________________


GAMMA = 1
env = mo_gym.MORecordEpisodeStatistics(mo_gym.make("env/SelectService-pcn", preprocessed_data_dir=MultiCloud_data_dir,service_querry=serviceIds ), gamma=GAMMA)
eval_env = mo_gym.make("env/SelectService-pcn", preprocessed_data_dir=MultiCloud_data_dir,service_querry=serviceIds )


KeyboardInterrupt: 

#### Test env with random actions just to make sure there is no bug in env and the MDP works fine

In [ ]:

nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: ",obs)
while (terminated == False):
    old_obs = obs
    action = env.action_space.sample()  # this is where you would insert your policy
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs:  ",old_obs,"\naction :", action,"\nnew obs: ",obs,"\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

#print("Selected services : ", env.composition.print_composition())

init obs:  [0 0 0]
old obs:   [0 0 0] 
action : 3 
new obs:  [8 0 0] 
reward : [0 0 0 0 0 0] 	erminated : False
_____________________________________________________
old obs:   [8 0 0] 
action : 0 
new obs:  [9 0 0] 
reward : [0 0 0 0 0 0] 	erminated : False
_____________________________________________________
old obs:   [9 0 0] 
action : 3 
new obs:  [17  0  0] 
reward : [0 0 0 0 0 0] 	erminated : False
_____________________________________________________
old obs:   [17  0  0] 
action : 0 
new obs:  [18  0  0] 
reward : [0 0 0 0 0 0] 	erminated : False
_____________________________________________________
old obs:   [18  0  0] 
action : 2 
new obs:  [18  0  0] 
reward : [-1 -1 -1 -1 -1 -1] 	erminated : True
_____________________________________________________
acc_rew: [-1 -1 -1 -1 -1 -1]


In [ ]:
pf = env.unwrapped.pareto_front()
print(pf)

Pareto already exist for [0, 1, 2] in multi cloud of 20 cloud : True
get pareto from the file : ./src/data/paretos/20clouds/s0s1s2.csv
[array([1.0719, 1.827 , 1.5331, 2.5861, 1.9563, 2.    ]), array([0.8676, 1.5422, 1.1235, 2.7382, 2.607 , 2.    ]), array([1.9927, 1.2623, 0.7643, 1.4776, 1.9304, 3.    ]), array([0.853 , 1.2719, 2.7767, 1.5664, 2.2773, 1.    ]), array([2.2984, 2.3596, 1.3301, 1.8413, 1.428 , 2.    ]), array([1.8425, 2.6836, 2.705 , 1.2183, 0.9251, 1.    ]), array([1.552 , 2.3654, 2.7266, 0.8884, 2.4476, 2.    ]), array([1.2021, 1.1889, 2.6686, 1.9665, 1.6158, 2.    ]), array([1.2555, 2.4035, 2.1465, 2.1907, 1.5157, 2.    ]), array([2.693 , 0.9774, 1.8098, 1.8033, 0.8691, 1.    ]), array([1.1078, 1.79  , 1.0949, 2.5846, 2.6852, 1.    ]), array([2.5992, 2.0686, 1.9899, 1.8775, 0.9694, 1.    ]), array([1.3889, 2.0924, 2.9811, 2.5236, 0.9307, 1.    ]), array([2.462 , 2.1281, 2.2159, 1.8972, 0.5937, 2.    ]), array([2.5689, 1.7913, 2.3959, 1.9835, 0.3943, 1.    ]), array([0.

In [ ]:
agent = PCN(

        env,
        scaling_factor=np.array([1,1, 1,1, 1,1, 1]),
        learning_rate=0.01,
        batch_size=256,
        project_name="Paper_test",
        experiment_name="PCN_s0s1s2s3s4",
        log=True,
    )

wandb: WARNING WANDB_NOTEBOOK_NAME should be a path to a notebook file, couldn't find envelope_minecart.
wandb: Currently logged in as: nabilachehlafekir. Use `wandb login --relogin` to force relogin


In [ ]:
agent.train(
        eval_env=eval_env,
        total_timesteps=10000,
        ref_point=np.array([-1 ,-1 ,-1 ,-1 ,-1 , -1]),
        num_er_episodes=20,
        max_buffer_size=50,
        num_model_updates=50,
        max_return=np.array(Constant.number_objectives*[number_services]),
        known_pareto_front=pf,

    )

c:\Users\jn_fe\anaconda3\lib\site-packages\morl_baselines\multi_policy\pcn\pcn.py:219: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_new.cpp:278.)
  th.tensor(obs).to(self.device),


step 161 	 return [ 0.1977  0.3905  0.3178  0.2712  0.3391 -0.2   ], ([1.1893 1.349  1.2585 1.3513 1.2254 0.9798]) 	 loss 1.449E+00 	 horizons 5.0
step 210 	 return [1.7158 1.5458 1.0234 1.1258 1.2191 1.3   ], ([0.999  0.8896 0.7817 0.8659 0.8022 0.9   ]) 	 loss 1.229E+00 	 horizons 4.9
step 257 	 return [2.0859 1.7777 1.1818 1.6539 1.6214 1.7   ], ([0.4473 0.3009 0.27   0.4302 0.3915 0.4583]) 	 loss 1.058E+00 	 horizons 4.7
step 296 	 return [2.5783 2.0546 0.9116 1.4108 1.0976 2.1   ], ([0.3617 0.1763 0.1987 0.2983 0.3551 0.3   ]) 	 loss 9.159E-01 	 horizons 3.9
step 337 	 return [2.3152 2.1451 1.0712 1.7485 1.0635 1.9   ], ([0.4212 0.1152 0.2684 0.4205 0.3129 0.3   ]) 	 loss 6.919E-01 	 horizons 4.1
step 377 	 return [2.8177 2.1353 0.7807 1.5431 0.7886 2.    ], ([0.2601 0.0932 0.0459 0.1678 0.1246 0.    ]) 	 loss 4.686E-01 	 horizons 4.0
step 417 	 return [2.341  2.0593 1.1542 1.625  1.0581 2.    ], ([0.3724 0.0673 0.1794 0.3168 0.2783 0.    ]) 	 loss 2.542E-01 	 horizons 4.0
step 46

# Use the trained agent with diffrent prefrences  :

In [ ]:

nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: ",obs)
while (terminated == False):
    old_obs = obs
    action = agent.eval( obs =obs, w = [1/6,1/6,1/6,1/6,1/6,1/6])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs:  ",old_obs,"\naction :", action,"\nnew obs: ",obs,"\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)
print("Selected services : ", env.composition.print_composition())


In [ ]:

# Second prefrences :
nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: service [",obs//nb_clouds ,"cloud ",obs%nb_clouds,"]")
while (terminated == False):
    old_obs = obs
    action = mp_moql.eval( obs = obs, w = [1/4,1/4,1/4,0,1/4,0])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs: [ service",old_obs//nb_clouds,"cloud",old_obs%nb_clouds,"]\naction :", action,"\nnew obs: [ service",obs//nb_clouds,"cloud",obs%nb_clouds,"]\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

print("Selected services : ", env.composition.print_composition())

In [ ]:

# Second prefrences :
nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: service [",obs//nb_clouds ,"cloud ",obs%nb_clouds,"]")
while (terminated == False):
    old_obs = obs
    action = mp_moql.eval( obs = obs, w = [1/2,1/4,1/4,0,0,0])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs: [ service",old_obs//nb_clouds,"cloud",old_obs%nb_clouds,"]\naction :", action,"\nnew obs: [ service",obs//nb_clouds,"cloud",obs%nb_clouds,"]\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

print("Selected services : ", env.composition.print_composition())